# Phase 5 — Evaluation

**Weeks 13–15 · Goal:** Measure retrieval and answer quality on a golden Q&A set with HTML reports and regression baselines.

## What you will learn

- **Golden set** — `data/benchmarks/golden_qa.json` (15 samples across sample + long docs)
- **Retrieval metrics** — page recall, chunk-type recall, keyword recall
- **Answer metrics** — faithfulness heuristics, keyword overlap, optional RAGAS
- **HTML reports** — human-readable eval summaries in `phases/phase_05_evaluation/reports/`
- **Baselines** — compare runs and alert on regressions

> Run retrieval-only eval without an LLM — ideal for CI.


In [1]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


Project root: C:\Users\dyh\Desktop\RAG


In [2]:
import json

golden_path = ROOT / "data/benchmarks/golden_qa.json"
samples = json.loads(golden_path.read_text(encoding="utf-8"))
print(f"Golden set: {len(samples)} samples")
tags = {}
for s in samples:
    tags[s.get("tag", "untagged")] = tags.get(s.get("tag", "untagged"), 0) + 1
print("By tag:", tags)
print("\nExample sample:")
print(json.dumps(samples[0], indent=2)[:500], "...")


Golden set: 17 samples
By tag: {'untagged': 17}

Example sample:
{
  "id": "sr-text-revenue",
  "question": "What was Acme Corp total revenue in fiscal year 2024?",
  "ground_truth_answer": "Total revenue was $191.7M in fiscal year 2024, representing 18.4% year-over-year growth.",
  "doc_path": "data/raw/sample_report.pdf",
  "ground_truth_doc_ids": [
    "ed7d53f9b08caa39"
  ],
  "expected_pages": [
    1
  ],
  "tags": [
    "text",
    "sample-report"
  ]
} ...


In [3]:
from phases.phase_05_evaluation.metrics import compute_sample_metrics, sample_passed
from shared.models import EvalResult, MetricScore

# Illustrative metric computation on a fake result
fake = EvalResult(
    sample_id="demo",
    query="What was Q4 revenue?",
    generated_answer="",
    ground_truth_answer="$191.7M",
    retrieved_chunk_ids=["c1"],
    metrics=[
        MetricScore(name="retrieval_page_recall", value=1.0),
        MetricScore(name="latency_ms", value=95.0),
    ],
    passed=True,
    latency_ms=95.0,
)
print("Sample passed:", sample_passed(fake.metrics, require_answer=False))


Sample passed: False


## Run full evaluation

```bash
python phases/phase_05_evaluation/run_full_eval.py \
  --system phase_02 --retrieve-only --hybrid --recursive-chunk \
  --ingest-if-missing --tag long-doc
```

Open the generated HTML report under `phases/phase_05_evaluation/reports/`.

**Next:** [06_production_api.ipynb](06_production_api.ipynb)
